In [12]:
from sklearn.model_selection import KFold, StratifiedKFold, StratifiedGroupKFold
import pandas as pd
import numpy as np

In [14]:
def set_folds(df, X, y, groups, group_by_column_name, fold_col_name, n_splits=10, random_state=42):
    train_folds = []
    test_folds = []
    sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    for i, (train_idx, test_idx) in enumerate(sgkf.split(np.arange(len(X)),np.round(y), groups)):
        df.loc[df[group_by_column_name].isin(X[test_idx]), fold_col_name] = i
        train_folds.append(train_idx)
        test_folds.append(test_idx)
    df[fold_col_name] = df[fold_col_name].astype(int)
    return train_folds, test_folds


def verify_folds(train_folds, test_folds, groups, N_in_groups, n_splits=10):
    s = set()
    counts = []
    for i in range(n_splits):
        # check there is no overlap between train and test user_ids
        x = np.intersect1d(np.unique(groups[train_folds[i]]),np.unique(groups[test_folds[i]]))
        assert len(x) == 0
        # check there is no intersection between user_ids across test folds
        assert len(s.intersection(groups[test_folds[i]])) == 0
        s = s.union((groups[test_folds[i]]))
        counts.append(len(np.unique(groups[test_folds[i]])))
    assert sum(counts) == N_in_groups

In [15]:
n_splits = 10

## Dataset: DS4UD

### User-level representation

In [66]:
group_by_column_name = 'user_id'
outcome_path = ""
affect_column_name = 'avg_from_wave_affect'
energy_column_name = 'avg_from_wave_energy'

df_affect = pd.read_pickle(outcome_path)[[group_by_column_name, affect_column_name]].dropna(subset=[affect_column_name])
df_energy = pd.read_pickle(outcome_path)[[group_by_column_name, energy_column_name]].dropna(subset=[energy_column_name])
len(df_affect), len(df_energy)

(120, 120)

#### Outcome: Valence

In [67]:
df = df_affect
X = df.user_id.values
y = df[affect_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_affect', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

# df_affect.to_csv("", index=False)

     user_id  avg_from_wave_affect  folds_affect
0         16                2.0000             6
1         27                3.0000             8
2         32                3.0000             0
3         53                1.7500             1
4         70                2.2000             2
..       ...                   ...           ...
115     2180                3.0000             3
116     2216                1.2500             5
117     2228                2.2500             7
118     2287                2.4000             3
119     2306                3.3333             4

[120 rows x 3 columns]


/cronus_data/pchitale/miniconda3/envs/vpy38/lib/python3.8/site-packages/sklearn/model_selection/_split.py:950: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=10.
  warnings.warn(


#### Outcome: Arousal

In [68]:
df = df_energy
X = df.user_id.values
y = df[energy_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_energy', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

# df_energy.to_csv("", index=False)

     user_id  avg_from_wave_energy  folds_energy
0         16                0.3333             7
1         27                0.2500             0
2         32                1.0000             5
3         53                1.0000             1
4         70                1.0000             2
..       ...                   ...           ...
115     2180                1.0000             8
116     2216                0.5000             9
117     2228                0.2500             1
118     2287                0.6000             2
119     2306                1.0000             7

[120 rows x 3 columns]


/cronus_data/pchitale/miniconda3/envs/vpy38/lib/python3.8/site-packages/sklearn/model_selection/_split.py:950: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=10.
  warnings.warn(


### Wave-level representation

In [78]:
## user_wave
group_by_column_name = 'user_wave_id'
outcome_path = ""
affect_column_name = 'avg_from_ema_affect'
energy_column_name = 'avg_from_ema_energy'

df_affect = pd.read_pickle(outcome_path)[['user_id', group_by_column_name, affect_column_name]].dropna(subset=[affect_column_name])
df_energy = pd.read_pickle(outcome_path)[['user_id', group_by_column_name, energy_column_name]].dropna(subset=[energy_column_name])
len(df_affect), len(df_energy)

(406, 406)

#### Outcome: Valence

In [79]:
df = df_affect
X = df[group_by_column_name].values
y = df[affect_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_affect', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

df_affect.to_csv("", index=False)

406

#### Outcome: Arousal

In [80]:
df = df_energy
X = df[group_by_column_name].values
y = df[energy_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_energy', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

df_energy.to_csv("", index=False)

/cronus_data/pchitale/miniconda3/envs/vpy38/lib/python3.8/site-packages/sklearn/model_selection/_split.py:950: UserWarning: The least populated class in y has only 9 members, which is less than n_splits=10.
  warnings.warn(


406

### Document-level representation

In [81]:
## user_wave
group_by_column_name = 'message_id'
outcome_path = ""
affect_column_name = 'affect'
energy_column_name = 'energy'

df_affect = pd.read_pickle(outcome_path)[['user_id', group_by_column_name, affect_column_name]].dropna(subset=[affect_column_name])
df_energy = pd.read_pickle(outcome_path)[['user_id', group_by_column_name, energy_column_name]].dropna(subset=[energy_column_name])
len(df_affect), len(df_energy)

(10108, 10108)

#### Outcome: Valence

In [82]:
df = df_affect
X = df[group_by_column_name].values
y = df[affect_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_affect', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

df_affect.to_csv("", index=False)

10108

#### Outcome: Arousal

In [83]:
df = df_energy
X = df[group_by_column_name].values
y = df[energy_column_name].values
groups = df.user_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_energy', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.user_id.nunique(), n_splits)

df_energy.to_csv("", index=False)

10108

## Wassa 2023 & 2024

### User-level representation

In [16]:
group_by_column_name = 'person_id'
outcome_path = "datasets/wassa/outcomes_user_wassa34.pkl"
empathy_column_name = 'person_empathy'
distress_column_name = 'person_distress'

df_empathy = pd.read_pickle(outcome_path)[[group_by_column_name, empathy_column_name]].dropna(subset=[empathy_column_name])
df_distress = pd.read_pickle(outcome_path)[[group_by_column_name, distress_column_name]].dropna(subset=[distress_column_name])
len(df_empathy), len(df_distress)

(180, 180)

#### Outcome: Empathy

In [21]:
df = df_empathy
X = df[group_by_column_name].values
y = df[empathy_column_name].values
groups = df.person_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_empathy', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.person_id.nunique(), n_splits)

df_empathy.to_csv("", index=False)

/cronus_data/pchitale/miniconda3/envs/vpy38/lib/python3.8/site-packages/sklearn/model_selection/_split.py:950: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=10.
  warnings.warn(


180

#### Outcome: Arousal

In [29]:
df = df_distress
X = df[group_by_column_name].values
y = df[distress_column_name].values
groups = df.person_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_distress', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.person_id.nunique(), n_splits)

df_distress.to_csv("", index=False)

180

### Document-level representation

In [ ]:
## user
group_by_column_name = 'message_id'
outcome_path = "/datasets/wassa/w23tr24trd.pkl"
empathy_column_name = 'person_empathy'
distress_column_name = 'person_distress'

df_empathy = pd.read_pickle(outcome_path)[['person_id', group_by_column_name, empathy_column_name]].dropna(subset=[empathy_column_name])
df_distress = pd.read_pickle(outcome_path)[['person_id', group_by_column_name, distress_column_name]].dropna(subset=[distress_column_name])
len(df_empathy), len(df_distress)

#### Outcome: Empathy

In [39]:
df = df_empathy
X = df[group_by_column_name].values
y = df[empathy_column_name].values
groups = df.person_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_empathy', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.person_id.nunique(), n_splits)

df_empathy.to_csv("", index=False)

1837

#### Outcome: Distress

In [40]:
df = df_distress
X = df[group_by_column_name].values
y = df[distress_column_name].values
groups = df.person_id.values
train_folds, test_folds = set_folds(df, X, y, groups, group_by_column_name, fold_col_name='folds_distress', n_splits=n_splits, random_state=42)
verify_folds(train_folds, test_folds, groups, df.person_id.nunique(), n_splits)

df_distress.to_csv("", index=False)

1837